# FastAPI API Functionality Test

Learn and test the REST API layer in `src/api/app.py` and `src/api/middleware.py`.

Covered functionality:

- FastAPI app configuration
- request/response models
- rate limiter setup
- `/health`
- `/query`
- `/query/stream` SSE response shape
- `/history/{thread_id}`
- `/agents/status`
- Teams router mounting

In [ ]:
from pathlib import Path
import ast
import json
import sys

cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / 'src' / 'api' / 'app.py').exists()), cwd)
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

api_path = project_root / 'src' / 'api' / 'app.py'
middleware_path = project_root / 'src' / 'api' / 'middleware.py'
source = api_path.read_text(encoding='utf-8', errors='replace')
middleware_source = middleware_path.read_text(encoding='utf-8', errors='replace')
lines = source.splitlines()
tree = ast.parse(source)

print('project_root:', project_root)
print('api_path:', api_path)
print('middleware_path:', middleware_path)
print('api_lines:', len(lines))

In [ ]:
def show_lines(start, end):
    end = min(end, len(lines))
    for n in range(start, end + 1):
        print(f'{n:4d}: {lines[n-1]}')

print('Top-level defs/classes:')
for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
        print(f'{node.lineno:4d}-{getattr(node, "end_lineno", node.lineno):4d}', type(node).__name__, node.name)

print('\nRoute decorators:')
for i, line in enumerate(lines, 1):
    if line.strip().startswith('@app.'):
        print(f'{i:4d}: {line.strip()}')

## 1. Request and Response Models

In [ ]:
show_lines(34, 58)

assert 'class QueryRequest' in source
assert 'class QueryResponse' in source
assert 'approved' in source
assert 'pending_action' in source
print('PASS: API models include HITL fields')

## 2. Rate Limiting Middleware

`limiter` uses IP-based limits for normal API clients. `user_limiter` keys Teams traffic by `X-User-Id` so one Teams IP does not throttle the entire org.

In [ ]:
print(middleware_source)

from api.middleware import get_user_id, limiter, user_limiter

class FakeRequest:
    def __init__(self, headers):
        self.headers = headers
        self.client = type('Client', (), {'host': '127.0.0.1'})()

assert get_user_id(FakeRequest({'X-User-Id': 'u123'})) == 'u123'
print('limiter:', limiter)
print('user_limiter:', user_limiter)

## 3. Import App and List Routes

If this fails, the missing dependency or graph/checkpointer issue is shown clearly.

In [ ]:
api_import_error = None
try:
    from api.app import app, QueryRequest, QueryResponse
    print('app:', app)
    print('routes:')
    for route in app.routes:
        methods = sorted(getattr(route, 'methods', []) or [])
        print(methods, getattr(route, 'path', '?'))
except Exception as exc:
    api_import_error = exc
    print('API import failed:', repr(exc))

## 4. Model Validation Without Running The Server

In [ ]:
if api_import_error is None:
    body = QueryRequest(query='Who owns the bookings dataset?', data_products=['bookings'], approved=False)
    print(body)
    response = QueryResponse(
        query_id='q1', thread_id='t1', intent='governance', summary='ok',
        confidence=0.9, sources=[], auto_tickets=[], anomalies=[], errors=[],
        execution_ms=123.0, pending_action=None,
    )
    print(response)
else:
    print('Skipped because api.app import failed.')

## 5. Optional TestClient Smoke Test

This calls FastAPI in-process. It can trigger graph imports for query endpoints, so keep it small.

In [ ]:
RUN_TESTCLIENT = False

if RUN_TESTCLIENT and api_import_error is None:
    from fastapi.testclient import TestClient
    client = TestClient(app)
    r = client.get('/health')
    print('GET /health:', r.status_code, r.json())
    assert r.status_code == 200

    r = client.get('/agents/status')
    print('GET /agents/status:', r.status_code, r.json())
else:
    print('Skipped. Set RUN_TESTCLIENT = True after api.app imports cleanly.')